# NDVI Training Data Analysis — Construction Start Detection

Extracts **median NDVI**, **NDVI standard deviation** and **Moran's I**
for every training polygon across all available years (2015–2024),
analyses temporal patterns relative to the labeled construction start / end,
and trains a **Random Forest** classifier to identify the best features
for detecting the start of construction.

**Training CSV columns (tab- or semicolon-separated):**
```
nhda_id | construction_start_year | construction_end_year | comment_start | comment_end
```

**Comment values – start:**
- *(empty)* → known start year  
- `already_under_construction` → site active before time series begins  
- `already_built_up` / `hard_to_say` → skip  
- `partially_built_up_already` → use normally  

**Comment values – end:**
- *(empty)* → known end year  
- `not_finished` / `probably_not_finished` / `no_end_visible` / `hard_to_say` → right-censored  

**Outputs:** `training_ndvi_timeseries.csv`, feature CSVs, threshold CSVs, plots.

In [ ]:
import geopandas as gpd
import rasterio
from rasterio.mask import mask as rio_mask
from rasterio.transform import array_bounds
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shapely.geometry
import re
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

# =============================================================================
# CONFIGURATION  –  adjust paths before running
# =============================================================================

INPUT_GPKG   = r'C:\Users\agz90fk\Documents\Masterarbeit\03_Daten\Output\Analysis_w_LoD2\New_Housing_Development_Areas\NHDA_residential_wsf2015_max10pct.gpkg'
# TRAINING_CSV_ORIGINAL = r"C:\Users\agz90fk\Documents\Masterarbeit\03_Daten\Output\Analysis_w_LoD2\Construction_Time_Estimate\training_sample_100_nhda.csv"
# TRAINING_CSV_NEW      = r"C:\Users\agz90fk\Documents\Masterarbeit\03_Daten\Output\Analysis_w_LoD2\Construction_Time_Estimate\training_sample_low_confidence_20.csv"  # ← dein neues CSV
TRAINING_CSV          = r"C:\Users\agz90fk\Documents\Masterarbeit\03_Daten\Output\Analysis_w_LoD2\Construction_Time_Estimate\training_sample_combined.csv"
NDVI_DIR     = r'C:\Users\agz90fk\Documents\Masterarbeit\03_Daten\Output\DatasetSpecific\Sentinel_2_WASP_v2\NDVI_Bavaria\Bayern_Final\masked'
NDVI_PATTERN = 'Bayern_NDVI_*median_25832.tif'
OUTPUT_DIR   = r'C:\Users\agz90fk\Documents\Masterarbeit\03_Daten\Output\Analysis_w_LoD2\Construction_Time_Estimate'

# Window sizes for aligned plots and feature computation
WINDOW_BEFORE    = 8   # years before anchor to show in aligned plots
WINDOW_AFTER     = 8   # years after anchor
SUSTAINED_WINDOW = 3   # years after start used in sustained-NDVI check
STABLE_WINDOW    = 2   # years after end used in stability check
DPI              = 150


In [ ]:
def load_training_csv(csv_path):
    """
    Parse training CSV (tab- or semicolon-separated) with start and end labels.

    Returns a DataFrame with columns:
      nda_id, start_year, end_year,
      comment_start, comment_end,
      category_start, category_end
    """
    import csv as _csv

    with open(csv_path, encoding='utf-8-sig') as f:
        lines = f.read().splitlines()

    # Find header row
    header_idx = next(
        i for i, line in enumerate(lines)
        if 'nhda_id' in line or 'nda_id' in line
    )
    reader = _csv.DictReader(lines[header_idx:], delimiter=';')
    df = pd.DataFrame(list(reader))

    # Drop blank rows (Excel artefact)
    id_col = 'nhda_id' if 'nhda_id' in df.columns else 'nda_id'
    df = df[df[id_col].notna() & (df[id_col].str.strip() != '')].reset_index(drop=True)
    print(f'  CSV rows after cleanup: {len(df)}')

    # Normalise column names
    df.columns = df.columns.str.strip()
    if 'nhda_id' in df.columns:
        df = df.rename(columns={'nhda_id': 'nda_id'})

    df['start_year'] = pd.to_numeric(df.get('construction_start_year', np.nan), errors='coerce')
    df['end_year']   = pd.to_numeric(df.get('construction_end_year',   np.nan), errors='coerce')

    df['comment_start'] = (df.reindex(columns=['comment_start'])['comment_start']
                           .fillna('').str.strip().str.lower())
    df['comment_end']   = (df.reindex(columns=['comment_end'])['comment_end']
                           .fillna('').str.strip().str.lower())

    # ── START categories ──────────────────────────────────────────────────
    df['category_start'] = 'labeled'
    df.loc[df['comment_start'].isin({'already_built_up'}),
           'category_start'] = 'skip'
    df.loc[df['comment_start'] == 'already_under_construction',
           'category_start'] = 'under_construction'
    df.loc[df['comment_start'] == 'partially_built_up_already',
           'category_start'] = 'partially_built_up'

    # ── END categories ────────────────────────────────────────────────────
    skip_end = {'not_finished', 'probably_not_finished', 'already_finished'}
    df['category_end'] = 'labeled'
    df.loc[df['end_year'].isna(),            'category_end'] = 'unknown'
    df.loc[df['comment_end'].isin(skip_end), 'category_end'] = 'skip'

    return df


In [ ]:
def build_ndvi_tile_index(ndvi_dir, pattern):
    """Return {year: [Path, ...]} from NDVI raster files matching *pattern*."""
    tile_index = {}
    for f in sorted(Path(ndvi_dir).glob(pattern)):
        m = re.search(r'(\d{4})', f.stem)
        if m:
            year = int(m.group(1))
            tile_index.setdefault(year, []).append(f)
    for year, paths in sorted(tile_index.items()):
        print(f'  {year}: {len(paths)} tile(s)  →  {[p.name for p in paths]}')
    return tile_index


In [ ]:
def extract_ndvi_stats(geometry, raster_paths, geom_crs):
    """
    Extract per-polygon NDVI statistics from a list of raster tiles.

    Returns a dict with keys: median, std, iqr, local_var, morans_i
    or None if no valid pixels are found.
    """
    if isinstance(raster_paths, (str, Path)):
        raster_paths = [raster_paths]

    for raster_path in raster_paths:
        try:
            with rasterio.open(raster_path) as src:
                # Reproject geometry to raster CRS if needed
                geom = geometry
                if geom_crs != src.crs:
                    geom = gpd.GeoSeries([geometry], crs=geom_crs).to_crs(src.crs).iloc[0]

                # Skip if polygon does not intersect this tile
                raster_box = shapely.geometry.box(
                    *array_bounds(src.height, src.width, src.transform)
                )
                if not geom.intersects(raster_box):
                    continue

                # Mask raster to polygon extent
                nodata = src.nodata
                fill   = nodata if nodata is not None else -9999
                out_image, _ = rio_mask(
                    src, [geom], crop=True, nodata=fill, all_touched=True
                )

                arr = out_image[0].astype(np.float32)

                # Build valid-pixel mask (NDVI scaled -100 to 100, not -1 to 1)
                valid_mask = (arr >= -100.0) & (arr <= 100.0)
                if nodata is not None:
                    if np.isnan(nodata):
                        valid_mask &= ~np.isnan(arr)
                    else:
                        valid_mask &= (arr != nodata)
                arr   = np.where(valid_mask, arr, np.nan)
                valid = arr[~np.isnan(arr)]
                
                # Scale back to -1 to 1 range
                valid = valid / 100.0

                if valid.size < 10:
                    return None

                # ── Moran's I (vectorised, 4-neighbour) ──────────────────
                def morans_i(a):
                    x    = a.copy()
                    mask = ~np.isnan(x)
                    if mask.sum() < 10:
                        return np.nan
                    mean  = np.nanmean(x)
                    dev   = x - mean
                    right = np.roll(dev, -1, axis=1)
                    down  = np.roll(dev, -1, axis=0)
                    mask_r = mask & np.roll(mask, -1, axis=1)
                    mask_d = mask & np.roll(mask, -1, axis=0)
                    num = (
                        np.nansum(dev[mask_r] * right[mask_r]) +
                        np.nansum(dev[mask_d] * down[mask_d])
                    )
                    w   = mask_r.sum() + mask_d.sum()
                    den = np.nansum(dev[mask] ** 2)
                    if den == 0 or w == 0:
                        return np.nan
                    return float((mask.sum() / w) * (num / den))

                # ── Local variance (mean squared neighbour difference) ────
                def neighbor_diff_sq(a):
                    diffs = []
                    for shift in [(0, 1), (1, 0), (1, 1), (1, -1)]:
                        shifted = np.roll(a, shift, axis=(0, 1))
                        m = ~np.isnan(a) & ~np.isnan(shifted)
                        if m.sum() > 0:
                            diffs.append(np.mean((a[m] - shifted[m]) ** 2))
                    return float(np.mean(diffs)) if diffs else np.nan

                return {
                    'median':    float(np.nanmedian(valid)),
                    'std':       float(np.nanstd(valid)),
                    'iqr':       float(np.nanpercentile(valid, 75) - np.nanpercentile(valid, 25)),
                    'local_var': neighbor_diff_sq(arr),
                    'morans_i':  morans_i(arr),
                }

        except Exception:
            continue

    return None


def extract_all_time_series(gdf, ndvi_tile_index):
    """Return a long-format DataFrame with NDVI stats per NDA per year."""
    years   = sorted(ndvi_tile_index.keys())
    crs     = gdf.crs
    records = []
    n       = len(gdf)
    print(f'  Extracting NDVI for {n} NDAs x {len(years)} years...')
    for i, (_, row) in enumerate(gdf.iterrows()):
        if i % 10 == 0:
            print(f'    [{i + 1}/{n}]')
        for year in years:
            stats = extract_ndvi_stats(row.geometry, ndvi_tile_index[year], crs)
            records.append({
                'nda_id':    row['nda_id'],
                'year':      year,
                'median':    stats['median']    if stats else np.nan,
                'std':       stats['std']       if stats else np.nan,
                'iqr':       stats['iqr']       if stats else np.nan,
                'local_var': stats['local_var'] if stats else np.nan,
                'morans_i':  stats['morans_i']  if stats else np.nan,
            })
    return pd.DataFrame(records)


In [ ]:
def plot_heatmaps(ts_df, label_map, null_ids, output_dir):
    """Plot median NDVI and NDVI STD heatmaps sorted by construction start year."""
    print('  Building heatmaps...')
    years       = sorted(ts_df['year'].unique())
    known_ids   = sorted(label_map.keys(), key=lambda x: label_map[x])
    null_sorted = sorted(null_ids & set(ts_df['nda_id'].unique()))
    ordered_ids = known_ids + null_sorted

    median_mat = np.full((len(ordered_ids), len(years)), np.nan)
    std_mat    = np.full((len(ordered_ids), len(years)), np.nan)
    year_idx   = {y: j for j, y in enumerate(years)}

    for i, nda_id in enumerate(ordered_ids):
        for _, row in ts_df[ts_df['nda_id'] == nda_id].iterrows():
            j = year_idx.get(row['year'])
            if j is not None:
                median_mat[i, j] = row['median']
                std_mat[i, j]    = row['std']

    fig, axes = plt.subplots(1, 2, figsize=(18, max(8, len(ordered_ids) * 0.22)))
    fig.suptitle(
        'NDVI Time Series — Training NDAs\n(sorted by construction start year)',
        fontsize=13, fontweight='bold'
    )

    for ax, mat, title, cmap, vmin, vmax in [
        (axes[0], median_mat, 'Median NDVI', 'RdYlGn', -0.1, 0.7),
        (axes[1], std_mat,    'NDVI STD',    'YlOrRd',  0.0, 0.25),
    ]:
        im = ax.imshow(mat, aspect='auto', cmap=cmap, vmin=vmin, vmax=vmax,
                       interpolation='nearest')
        plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
        ax.set_xticks(range(len(years)))
        ax.set_xticklabels(years, rotation=45, ha='right', fontsize=8)
        ax.set_yticks(range(len(ordered_ids)))
        ax.set_yticklabels(ordered_ids, fontsize=5)
        ax.set_title(title, fontsize=11)
        ax.set_xlabel('Year', fontsize=10)
        ax.set_ylabel('NDA', fontsize=10)

        # Mark construction start year
        for i, nda_id in enumerate(ordered_ids):
            if nda_id in label_map and label_map[nda_id] in year_idx:
                ax.plot(year_idx[label_map[nda_id]], i,
                        marker='|', color='black', markersize=6, markeredgewidth=1.5)

        if null_sorted:
            sep = len(known_ids) - 0.5
            ax.axhline(sep, color='white', linewidth=1.5, linestyle='--')
            ax.text(0.01, (sep + 0.5) / len(ordered_ids), 'already_built_up ->',
                    transform=ax.transAxes, fontsize=7, color='white', va='bottom')

    plt.tight_layout()
    out = Path(output_dir) / 'heatmap_ndvi_training.png'
    plt.savefig(out, dpi=DPI, bbox_inches='tight')
    plt.close()
    print(f'  Saved: {out.name}')


In [ ]:
def _aligned_percentiles(aligned_dict):
    x, med, p10, p25, p75, p90 = [], [], [], [], [], []
    for o in sorted(aligned_dict.keys()):
        vals = [v for v in aligned_dict[o] if not np.isnan(v)]
        if len(vals) < 3:
            continue
        x.append(o)
        med.append(np.percentile(vals, 50))
        p10.append(np.percentile(vals, 10))
        p25.append(np.percentile(vals, 25))
        p75.append(np.percentile(vals, 75))
        p90.append(np.percentile(vals, 90))
    return np.array(x), np.array(med), np.array(p10), np.array(p25), np.array(p75), np.array(p90)


def plot_aligned_generic(ts_df, anchor_map, null_ids, window_before, window_after,
                         output_dir, filename, title_anchor_label):
    """
    Plot median NDVI and NDVI STD aligned to an anchor year (construction start or end).
    anchor_map: {nda_id: anchor_year}
    """
    print(f'  Building aligned plots ({title_anchor_label})...')
    offsets = range(-window_before, window_after + 1)

    aligned_median = {o: [] for o in offsets}
    aligned_std    = {o: [] for o in offsets}

    for nda_id, cy in anchor_map.items():
        sub = ts_df[ts_df['nda_id'] == nda_id].copy()
        sub['offset'] = sub['year'] - cy
        for _, row in sub.iterrows():
            o = int(row['offset'])
            if o in aligned_median:
                aligned_median[o].append(row['median'])
                aligned_std[o].append(row['std'])

    null_median_per_year, null_std_per_year = {}, {}
    for nda_id in null_ids:
        for _, row in ts_df[ts_df['nda_id'] == nda_id].iterrows():
            null_median_per_year.setdefault(row['year'], []).append(row['median'])
            null_std_per_year.setdefault(row['year'], []).append(row['std'])

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle(
        f'NDVI Patterns Relative to {title_anchor_label}\n'
        f'(0 = {title_anchor_label.lower()}, based on training data)',
        fontsize=13, fontweight='bold'
    )

    for ax, aligned_dict, null_dict_raw, ylabel, subplot_title, color in [
        (axes[0], aligned_median, null_median_per_year, 'Median NDVI', 'Median NDVI', '#2166AC'),
        (axes[1], aligned_std,    null_std_per_year,    'NDVI STD',    'NDVI STD',    '#D6604D'),
    ]:
        x, med, p10, p25, p75, p90 = _aligned_percentiles(aligned_dict)

        for o in sorted(aligned_dict.keys()):
            vals = [v for v in aligned_dict[o] if not np.isnan(v)]
            jitter = np.random.uniform(-0.1, 0.1, len(vals))
            ax.scatter([o + j for j in jitter], vals,
                       color=color, alpha=0.2, s=18, zorder=1)

        ax.fill_between(x, p10, p90, color=color, alpha=0.12, label='P10–P90')
        ax.fill_between(x, p25, p75, color=color, alpha=0.25, label='P25–P75')
        ax.plot(x, med, color=color, linewidth=2.5, zorder=3, label='Median')

        if 0 in aligned_dict:
            vals_at_0 = [v for v in aligned_dict[0] if not np.isnan(v)]
            if vals_at_0:
                ax.axhline(np.percentile(vals_at_0, 25), color=color,
                           linewidth=1, linestyle=':', alpha=0.7,
                           label=f"P25 at t=0: {np.percentile(vals_at_0, 25):.3f}")
                ax.axhline(np.percentile(vals_at_0, 75), color=color,
                           linewidth=1, linestyle='--', alpha=0.7,
                           label=f"P75 at t=0: {np.percentile(vals_at_0, 75):.3f}")

        if null_dict_raw:
            null_med = np.median([v for vlist in null_dict_raw.values() for v in vlist])
            ax.axhline(null_med, color='gray', linewidth=1.5, linestyle='-.',
                       alpha=0.7, label='already_built_up (median)')

        ax.axvline(0, color='black', linewidth=1.5, linestyle='--', alpha=0.8,
                   label=f'{title_anchor_label} (t=0)')
        ax.set_xlabel(f'Years relative to {title_anchor_label.lower()}', fontsize=11)
        ax.set_ylabel(ylabel, fontsize=11)
        ax.set_title(subplot_title, fontsize=11)
        ax.set_xticks(list(offsets))
        ax.set_xticklabels([str(o) for o in offsets])
        ax.grid(axis='y', linestyle='--', alpha=0.4)
        ax.spines[['top', 'right']].set_visible(False)
        ax.legend(fontsize=8, loc='best')

    plt.tight_layout()
    out = Path(output_dir) / filename
    plt.savefig(out, dpi=DPI, bbox_inches='tight')
    plt.close()
    print(f'  Saved: {out.name}')

    # Percentile table at anchor (offset = 0)
    print(f'\n  Percentile table at {title_anchor_label} (offset = 0):')
    print(f"  {'Metric':<14}  P10    P25    P50    P75    P90")
    for lbl, adict in [('Median NDVI', aligned_median), ('NDVI STD', aligned_std)]:
        vals = [v for v in adict.get(0, []) if not np.isnan(v)]
        if vals:
            print(f"  {lbl:<14}  " +
                  '  '.join([f'{np.percentile(vals, p):.3f}' for p in [10, 25, 50, 75, 90]]))


In [ ]:
def compute_thresholds(ts_df, start_map, end_map, null_ids, output_dir,
                        labeled_end=None):
    """Derive empirical detection thresholds from the training time series."""
    pcts        = [10, 25, 50, 75, 90]
    output_path = Path(output_dir)

    def pct(series, p):
        clean = series.dropna()
        return round(float(np.percentile(clean, p)), 4) if len(clean) > 0 else np.nan

    def r(v):
        return round(v, 4) if (v is not None and not np.isnan(v)) else np.nan

    # ── START features ────────────────────────────────────────────────────
    print('\n  Computing START features...')
    start_records = []
    for nda_id, cy in start_map.items():
        sub = ts_df[ts_df['nda_id'] == nda_id].sort_values('year').reset_index(drop=True)
        if len(sub) < 2:
            continue
        pre  = sub[sub['year'] <  cy]
        post = sub[sub['year'] >= cy]
        if len(pre) < 1 or len(post) < 1:
            continue

        row_pre_t = sub[sub['year'] == cy - 1]
        if len(row_pre_t) == 0:
            row_pre_t = sub[sub['year'] < cy].tail(1)
        row_at_t = sub[sub['year'] == cy]
        if len(row_at_t) == 0:
            row_at_t = sub[sub['year'] >= cy].head(1)
        if len(row_pre_t) == 0 or len(row_at_t) == 0:
            continue

        ndvi_drop  = float(row_pre_t['median'].values[0] - row_at_t['median'].values[0])
        pre_ndvi   = float(pre['median'].mean())
        post_win   = post[post['year'] <= cy + SUSTAINED_WINDOW]
        post_ndvi  = (float(post_win['median'].mean()) if len(post_win) > 0
                      else float(post['median'].iloc[0]))

        std_pre      = float(pre['std'].mean())
        post_std_win = post[post['year'] <= cy + 1]
        std_post     = (float(post_std_win['std'].mean()) if len(post_std_win) > 0
                        else float(post['std'].iloc[0]))

        pre_moran  = float(pre['morans_i'].mean())  if len(pre) > 0  else np.nan
        post_moran = float(post['morans_i'].mean()) if len(post) > 0 else np.nan
        moran_drop = (pre_moran - post_moran
                      if not np.isnan(pre_moran) and not np.isnan(post_moran) else np.nan)

        start_records.append({
            'nda_id':            nda_id,
            'construction_year': cy,
            'ndvi_drop':         r(ndvi_drop),
            'pre_ndvi':          r(pre_ndvi),
            'post_ndvi':         r(post_ndvi),
            'sustained_diff':    r(pre_ndvi - post_ndvi),
            'std_pre':           r(std_pre),
            'std_post':          r(std_post),
            'std_increase':      r(std_post - std_pre),
            'moran_pre':         r(pre_moran),
            'moran_post':        r(post_moran),
            'moran_drop':        r(moran_drop),
        })

    start_df = pd.DataFrame(start_records)
    print(f'  START features: {len(start_df)} NDAs')
    start_df.to_csv(output_path / 'training_ndvi_features_start.csv',
                    index=False, encoding='utf-8-sig')

    # ── END features ──────────────────────────────────────────────────────
# ── END features ──────────────────────────────────────────────────────
    print('\n  Computing END features...')
    end_records = []
    for nda_id, ey in end_map.items():
        sy  = start_map.get(nda_id)
        sub = (ts_df[ts_df['nda_id'] == nda_id]
               .sort_values('year').reset_index(drop=True))
        if len(sub) < 2 or sy is None:
            continue

        construction = sub[(sub['year'] >= sy) & (sub['year'] <= ey)]
        post         = sub[sub['year'] >  ey]
        pre          = sub[sub['year'] <  sy]

        if len(construction) == 0:
            continue

        std_peak        = float(construction['std'].max())
        std_at_end_rows = sub[sub['year'] == ey]
        std_at_end      = float(std_at_end_rows['std'].values[0]) if len(std_at_end_rows) > 0 else np.nan
        std_post_win    = post[post['year'] <= ey + 2]
        std_post_stable = float(std_post_win['std'].mean()) if len(std_post_win) > 0 else np.nan
        std_pre_base    = float(pre['std'].mean()) if len(pre) > 0 else np.nan
        std_drop        = (std_peak - std_post_stable
                           if not np.isnan(std_peak) and not np.isnan(std_post_stable) else np.nan)

        ndvi_at_end_rows = sub[sub['year'] == ey]
        ndvi_at_end      = float(ndvi_at_end_rows['median'].values[0]) if len(ndvi_at_end_rows) > 0 else np.nan
        ndvi_post_win    = post[post['year'] <= ey + 2]
        ndvi_post_stable = float(ndvi_post_win['median'].mean()) if len(ndvi_post_win) > 0 else np.nan
        ndvi_delta_max   = float(construction['median'].max() - construction['median'].min())

        moran_peak    = float(construction['morans_i'].max())
        moran_at_end  = float(sub[sub['year'] == ey]['morans_i'].values[0]) if len(std_at_end_rows) > 0 else np.nan
        moran_post    = float(post['morans_i'].mean()) if len(post) > 0 else np.nan

        end_records.append({
            'nda_id':           nda_id,
            'end_year':         ey,
            'std_peak':         r(std_peak),
            'std_at_end':       r(std_at_end),
            'std_drop':         r(std_drop),
            'std_post_stable':  r(std_post_stable),
            'std_pre_baseline': r(std_pre_base),
            'ndvi_at_end':      r(ndvi_at_end),
            'ndvi_post_stable': r(ndvi_post_stable),
            'ndvi_delta_max':   r(ndvi_delta_max),
            'moran_peak':       r(moran_peak),
            'moran_at_end':     r(moran_at_end),
            'moran_post':       r(moran_post),
        })

    end_df = pd.DataFrame(end_records)

    # ── Already_built_up profile ──────────────────────────────────────────
    null_records = []
    for nda_id in null_ids:
        sub = ts_df[ts_df['nda_id'] == nda_id].sort_values('year').reset_index(drop=True)
        if len(sub) == 0:
            continue
        null_records.append({
            'nda_id':    nda_id,
            'std_early': float(sub.head(2)['std'].mean()),
            'std_mean':  float(sub['std'].mean()),
            'ndvi_mean': float(sub['median'].mean()),
        })
    null_df = pd.DataFrame(null_records)
    print(f'  Already_built_up profiles: {len(null_df)}')

    # ── Print feature distributions ───────────────────────────────────────
    print('\n' + '=' * 60)
    print('FEATURE DISTRIBUTIONS — START (labeled NDAs)')
    print('=' * 60)
    dist_start_rows = []
    for metric in ['ndvi_drop', 'pre_ndvi', 'post_ndvi', 'sustained_diff',
                   'std_pre', 'std_post', 'std_increase',
                   'moran_pre', 'moran_post', 'moran_drop']:
        vals = start_df[metric].dropna() if metric in start_df.columns else pd.Series(dtype=float)
        if len(vals) == 0:
            continue
        row_d = {'metric': metric, 'mean': round(float(vals.mean()), 4)}
        for p in pcts:
            row_d[f'P{p}'] = round(float(np.percentile(vals, p)), 4)
        dist_start_rows.append(row_d)
    if dist_start_rows:
        dist_start_df = pd.DataFrame(dist_start_rows)[['metric', 'mean'] + [f'P{p}' for p in pcts]]
        print(dist_start_df.to_string(index=False))
    else:
        dist_start_df = pd.DataFrame(columns=['metric', 'mean'] + [f'P{p}' for p in pcts])
        print('  No valid START features found.')

    print('\n' + '=' * 60)
    print('FEATURE DISTRIBUTIONS — END (labeled NDAs)')
    print('=' * 60)
    dist_end_rows = []
    for metric in ['std_peak', 'std_at_end', 'std_drop', 'std_post_stable',
                   'std_pre_baseline', 'ndvi_at_end', 'ndvi_post_stable', 'ndvi_delta_max',
                   'moran_peak', 'moran_at_end', 'moran_post']:
        vals = end_df[metric].dropna() if metric in end_df.columns else pd.Series(dtype=float)
        if len(vals) == 0:
            continue
        row_d = {'metric': metric, 'mean': round(float(vals.mean()), 4)}
        for p in pcts:
            row_d[f'P{p}'] = round(float(np.percentile(vals, p)), 4)
        dist_end_rows.append(row_d)
    if dist_end_rows:
        dist_end_df = pd.DataFrame(dist_end_rows)[['metric', 'mean'] + [f'P{p}' for p in pcts]]
        print(dist_end_df.to_string(index=False))
    else:
        dist_end_df = pd.DataFrame(columns=['metric', 'mean'] + [f'P{p}' for p in pcts])
        print('  No valid END features found.')

    if len(null_df) > 0:
        print('\n' + '=' * 60)
        print('FEATURE DISTRIBUTIONS — already_built_up NDAs')
        print('=' * 60)
        for metric in ['std_early', 'std_mean', 'ndvi_mean']:
            vals = null_df[metric].dropna()
            print(f'  {metric:<12}  ' +
                  '  '.join([f'P{p}={np.percentile(vals, p):.3f}' for p in pcts]))

    # ── Duration analysis ─────────────────────────────────────────────────
    print('\n' + '=' * 60)
    print('CONSTRUCTION DURATION (end - start, labeled pairs only)')
    print('=' * 60)
    rows = []
    for nda_id, ey in end_map.items():
        sy = start_map.get(nda_id)
        if sy is not None:
            rows.append({'nda_id': nda_id, 'start': sy, 'end': ey, 'duration': ey - sy})
    paired = pd.DataFrame(rows)
    if len(paired) > 0:
        paired = paired[paired['duration'] >= 0]
    print(f'  N pairs: {len(paired)}')
    if len(paired) > 0:
        print(f'  Mean: {paired["duration"].mean():.1f} years  |  '
              f'Min: {paired["duration"].min()}  |  Max: {paired["duration"].max()}')
        for p in [10, 25, 50, 75, 90]:
            print(f'  P{p:2d}: {np.percentile(paired["duration"], p):.1f} years')
        print(paired['duration'].value_counts().sort_index().to_string())
    paired.to_csv(output_path / 'training_ndvi_duration_pairs.csv',
                  index=False, encoding='utf-8-sig')

    # ── Derive thresholds ─────────────────────────────────────────────────
    thresholds = {}
    thresholds['MIN_NDVI_DROP_P25']            = pct(start_df['ndvi_drop'],      25)
    thresholds['MIN_NDVI_DROP_P10']            = pct(start_df['ndvi_drop'],      10)
    thresholds['MIN_NDVI_DROP_P50']            = pct(start_df['ndvi_drop'],      50)
    thresholds['SUSTAINED_NDVI_THRESHOLD_P50'] = pct(start_df['post_ndvi'],      50)
    thresholds['SUSTAINED_NDVI_THRESHOLD_P75'] = pct(start_df['post_ndvi'],      75)
    thresholds['PRE_NDVI_MIN_P25']             = pct(start_df['pre_ndvi'],       25)
    thresholds['SUSTAINED_DIFF_P25']           = pct(start_df['sustained_diff'], 25)
    thresholds['SUSTAINED_DIFF_P50']           = pct(start_df['sustained_diff'], 50)
    thresholds['MIN_STD_INCREASE_P25']         = pct(start_df['std_increase'],   25)
    thresholds['MIN_STD_INCREASE_P50']         = pct(start_df['std_increase'],   50)

    if len(end_df) > 0:
        thresholds['END_STD_POST_STABLE_P50']  = pct(end_df['std_post_stable'],  50)
        thresholds['END_STD_POST_STABLE_P75']  = pct(end_df['std_post_stable'],  75)
        thresholds['END_STD_DROP_MIN_P10']     = pct(end_df['std_drop'],         10)
        thresholds['END_STD_DROP_MIN_P25']     = pct(end_df['std_drop'],         25)
        thresholds['END_STD_AT_END_P50']       = pct(end_df['std_at_end'],       50)
        thresholds['END_STD_AT_END_P75']       = pct(end_df['std_at_end'],       75)
        thresholds['END_NDVI_DELTA_MAX_P50']   = pct(end_df['ndvi_delta_max'],   50)
        thresholds['END_NDVI_DELTA_MAX_P75']   = pct(end_df['ndvi_delta_max'],   75)
        thresholds['END_NDVI_AT_END_P25']      = pct(end_df['ndvi_at_end'],      25)
        thresholds['END_NDVI_AT_END_P50']      = pct(end_df['ndvi_at_end'],      50)

    if len(null_df) > 0:
        thresholds['ALREADY_BUILT_STD_P25']    = pct(null_df['std_early'],  25)
        thresholds['ALREADY_BUILT_STD_P50']    = pct(null_df['std_early'],  50)
        thresholds['ALREADY_BUILT_NDVI_P25']   = pct(null_df['ndvi_mean'],  25)

    print('\n' + '=' * 60)
    print('THRESHOLD RECOMMENDATIONS')
    print('=' * 60)
    print('\n  ── START ──')
    for k, v in thresholds.items():
        if not k.startswith('END_') and not k.startswith('ALREADY_'):
            print(f'    {k:<45} = {v}')
    print('\n  ── END ──')
    for k, v in thresholds.items():
        if k.startswith('END_'):
            print(f'    {k:<45} = {v}')
    print('\n  ── GUARD ──')
    for k, v in thresholds.items():
        if k.startswith('ALREADY_'):
            print(f'    {k:<45} = {v}')

    # ── Save ──────────────────────────────────────────────────────────────
    all_dist = pd.concat(
        [dist_start_df.assign(domain='start'), dist_end_df.assign(domain='end')],
        ignore_index=True
    )
    all_dist.to_csv(output_path / 'training_ndvi_threshold_table.csv',
                    index=False, encoding='utf-8-sig')
    pd.DataFrame(list(thresholds.items()), columns=['threshold', 'value']).to_csv(
        output_path / 'training_ndvi_thresholds.csv',
        index=False, encoding='utf-8-sig'
    )
    print('\n  Saved: training_ndvi_features_start.csv')
    print('  Saved: training_ndvi_features_end.csv')
    print('  Saved: training_ndvi_duration_pairs.csv')
    print('  Saved: training_ndvi_threshold_table.csv')
    print('  Saved: training_ndvi_thresholds.csv')

    return thresholds, start_df, end_df


In [ ]:
def run_random_forest(ts_df, start_map, end_map, null_ids, output_dir):
    """
    Train a Random Forest classifier to distinguish construction-active years
    from pre-construction and post-construction years.

    Features per (polygon, year) row:
      median_ndvi, std_ndvi, morans_i, ndvi_change (delta from previous year)
      ndvi_rebound, ndvi_sustained_low

    Labels:
      0 = pre-construction / already_built_up
      1 = under construction  (start_year <= year <= end_year)

    Outputs:
      rf_hyperparameter_search.csv
      rf_hyperparameter_heatmap.png
      rf_feature_importance.png
      rf_confusion_matrix.png
      rf_classification_report.csv
    """
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.model_selection import GroupKFold, cross_val_predict, GridSearchCV
    from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
    from sklearn.inspection import permutation_importance

    output_path = Path(output_dir)

    # Fixed final model parameters chosen from grouped CV search
    final_n_estimators = 300
    final_max_depth = 10

    # ── Build labelled dataset ─────────────────────────────────────────────
    records = []
    for nda_id, sy in start_map.items():
        ey  = end_map.get(nda_id)          # may be None (censored)
        sub = (ts_df[ts_df['nda_id'] == nda_id]
               .sort_values('year').reset_index(drop=True))
        if len(sub) < 2:
            continue
        sub = sub.copy()
        sub['ndvi_change'] = sub['median'].diff()   # NaN for first year
        sub['ndvi_rebound'] = sub['median'].shift(-1) - sub['median']

        sustained = []
        medians = sub['median'].values
        for i in range(len(medians)):
            future = medians[i+1 : i+1+SUSTAINED_WINDOW]
            future_valid = future[~np.isnan(future)]
            sustained.append(float(np.mean(future_valid)) if len(future_valid) > 0 else np.nan)
        sub['ndvi_sustained_low'] = sustained
        for _, row in sub.iterrows():
            year = int(row['year'])
            # Under construction: start <= year; use end if known, else all post-start
            if ey is not None:
                label = 1 if sy <= year <= ey else 0
            else:
                label = 1 if year >= sy else 0
            records.append({
                'nda_id':       nda_id,
                'year':         year,
                'median_ndvi':  row['median'],
                'std_ndvi':     row['std'],
                'morans_i':     row['morans_i'],
                'ndvi_change':  row['ndvi_change'],
                'ndvi_rebound': row['ndvi_rebound'],
                'ndvi_sustained_low': row['ndvi_sustained_low'],
                'label':        label,
            })

    # Add already_built_up polygons as class 0
    for nda_id in null_ids:
        sub = (ts_df[ts_df['nda_id'] == nda_id]
               .sort_values('year').reset_index(drop=True))
        if len(sub) < 2:
            continue
        sub = sub.copy()
        sub['ndvi_change'] = sub['median'].diff()
        sub['ndvi_rebound'] = sub['median'].shift(-1) - sub['median']

        sustained = []
        medians = sub['median'].values
        for i in range(len(medians)):
            future = medians[i+1 : i+1+SUSTAINED_WINDOW]
            future_valid = future[~np.isnan(future)]
            sustained.append(float(np.mean(future_valid)) if len(future_valid) > 0 else np.nan)
        sub['ndvi_sustained_low'] = sustained

        for _, row in sub.iterrows():
            records.append({
                'nda_id':              nda_id,
                'year':                int(row['year']),
                'median_ndvi':         row['median'],
                'std_ndvi':            row['std'],
                'morans_i':            row['morans_i'],
                'ndvi_change':         row['ndvi_change'],
                'ndvi_rebound':        row['ndvi_rebound'],
                'ndvi_sustained_low':  row['ndvi_sustained_low'],
                'label':               0,
            })

    df_rf = pd.DataFrame(records)

    feature_cols = ['median_ndvi', 'std_ndvi', 'morans_i', 'ndvi_change', 'ndvi_rebound', 'ndvi_sustained_low']
    df_rf = df_rf.dropna(subset=feature_cols)

    print(f'\n  RF dataset: {len(df_rf)} rows  '
          f'(class 0: {(df_rf["label"]==0).sum()}, class 1: {(df_rf["label"]==1).sum()})')

    if len(df_rf) < 20 or df_rf['label'].nunique() < 2:
        print('  Not enough labelled data for Random Forest — skipping.')
        return

    X = df_rf[feature_cols].values
    y = df_rf['label'].values
    groups = df_rf['nda_id'].values

    # ── Hyperparameter search ───────────────────────────────────────────────
    cv = GroupKFold(n_splits=5)
    param_grid = {
        'n_estimators': [100, 200, 300, 500],
        'max_depth': [4, 6, 8, 10, None],
    }
    search = GridSearchCV(
        estimator=RandomForestClassifier(
            class_weight='balanced',
            random_state=42,
            n_jobs=1,
        ),
        param_grid=param_grid,
        scoring='f1_macro',
        cv=cv,
        n_jobs=1,
        refit=True,
        return_train_score=True,
    )
    search.fit(X, y, groups=groups)

    search_df = (pd.DataFrame(search.cv_results_)
                   [['param_n_estimators', 'param_max_depth',
                     'mean_test_score', 'std_test_score', 'rank_test_score']]
                   .rename(columns={
                       'param_n_estimators': 'n_estimators',
                       'param_max_depth': 'max_depth',
                       'mean_test_score': 'mean_cv_f1_macro',
                       'std_test_score': 'std_cv_f1_macro',
                       'rank_test_score': 'rank',
                   }))
    search_df['max_depth_label'] = search_df['max_depth'].astype(object).where(
        search_df['max_depth'].notna(), 'None'
    )
    depth_order = ['4', '6', '8', '10', 'None']
    search_df['max_depth_label'] = pd.Categorical(
        search_df['max_depth_label'].astype(str),
        categories=depth_order,
        ordered=True,
    )
    search_df = search_df.sort_values(['rank', 'n_estimators', 'max_depth_label'])
    search_df.to_csv(output_path / 'rf_hyperparameter_search.csv', index=False, encoding='utf-8-sig')

    heatmap_df = (search_df
                  .pivot(index='max_depth_label', columns='n_estimators', values='mean_cv_f1_macro')
                  .reindex(depth_order))

    fig, ax = plt.subplots(figsize=(7, 4.5))
    im = ax.imshow(heatmap_df.values, cmap='YlGnBu', aspect='auto')
    cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label('Mean CV F1-macro')
    ax.set_xticks(range(len(heatmap_df.columns)))
    ax.set_xticklabels([str(v) for v in heatmap_df.columns])
    ax.set_yticks(range(len(heatmap_df.index)))
    ax.set_yticklabels(heatmap_df.index.tolist())
    ax.set_xlabel('n_estimators')
    ax.set_ylabel('max_depth')
    ax.set_title('Random Forest Hyperparameter Search (GroupKFold)')

    for row_idx in range(len(heatmap_df.index)):
        for col_idx in range(len(heatmap_df.columns)):
            value = heatmap_df.iloc[row_idx, col_idx]
            if pd.notna(value):
                ax.text(col_idx, row_idx, f'{value:.3f}', ha='center', va='center', color='black', fontsize=9)

    plt.tight_layout()
    fig.savefig(output_path / 'rf_hyperparameter_heatmap.png', dpi=DPI, bbox_inches='tight')
    plt.close()

    best_params = search.best_params_
    print('\n  Hyperparameter search (5-fold GroupKFold, scoring = f1_macro):')
    print(search_df[['n_estimators', 'max_depth', 'mean_cv_f1_macro', 'std_cv_f1_macro', 'rank']].to_string(index=False))
    print(f"\n  Best parameters from search: n_estimators={best_params['n_estimators']}, "
          f"max_depth={best_params['max_depth']}")
    print(f"  Best mean CV F1-macro: {search.best_score_:.3f}")
    print(f'  Final model fixed to: n_estimators={final_n_estimators}, max_depth={final_max_depth}')
    print('  Saved: rf_hyperparameter_search.csv')
    print('  Saved: rf_hyperparameter_heatmap.png')

    # ── Cross-validated predictions with chosen final model ─────────────────
    clf = RandomForestClassifier(
        n_estimators=final_n_estimators,
        max_depth=final_max_depth,
        class_weight='balanced',
        random_state=42,
        n_jobs=1,
    )
    y_pred = cross_val_predict(clf, X, y, cv=cv, groups=groups, n_jobs=1)

    print('\n  Classification report (5-fold GroupKFold):')
    report_str = classification_report(
        y, y_pred,
        target_names=['pre/post-construction', 'under construction']
    )
    print(report_str)

    report_df = pd.DataFrame(
        classification_report(
            y, y_pred, output_dict=True,
            target_names=['pre/post-construction', 'under construction']
        )
    ).T
    report_df.to_csv(output_path / 'rf_classification_report.csv', encoding='utf-8-sig')

    # ── Confusion matrix ───────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(5, 4))
    ConfusionMatrixDisplay(
        confusion_matrix(y, y_pred),
        display_labels=['pre/post', 'construction']
    ).plot(ax=ax, colorbar=False)
    ax.set_title('Random Forest — Confusion Matrix (5-fold GroupKFold)', fontsize=11)
    plt.tight_layout()
    fig.savefig(output_path / 'rf_confusion_matrix.png', dpi=DPI, bbox_inches='tight')
    plt.close()
    print('  Saved: rf_confusion_matrix.png')

    # ── Feature importance ─────────────────────────────────────────────────
    clf.fit(X, y)   # fit on full dataset for importance scores
    perm_imp = permutation_importance(
        clf, X, y, n_repeats=20, random_state=42, n_jobs=1
    )
    imp_df = pd.DataFrame({
        'feature':            feature_cols,
        'importance_mean':    perm_imp.importances_mean,
        'importance_std':     perm_imp.importances_std,
        'gini_importance':    clf.feature_importances_,
    }).sort_values('importance_mean', ascending=False)

    print('\n  Permutation feature importance (full dataset):')
    print(imp_df.to_string(index=False))

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].barh(
        imp_df['feature'], imp_df['importance_mean'],
        xerr=imp_df['importance_std'], color='#2166AC', alpha=0.8
    )
    axes[0].set_title('Permutation Importance', fontsize=11)
    axes[0].set_xlabel('Mean accuracy decrease')
    axes[0].invert_yaxis()

    axes[1].barh(
        imp_df['feature'], imp_df['gini_importance'],
        color='#D6604D', alpha=0.8
    )
    axes[1].set_title('Gini Importance (MDI)', fontsize=11)
    axes[1].set_xlabel('Mean decrease in impurity')
    axes[1].invert_yaxis()

    fig.suptitle('Random Forest — Feature Importance', fontsize=13, fontweight='bold')
    plt.tight_layout()
    fig.savefig(output_path / 'rf_feature_importance.png', dpi=DPI, bbox_inches='tight')
    plt.close()
    print('  Saved: rf_feature_importance.png')
    print('  Saved: rf_classification_report.csv')

    # ── Save model ─────────────────────────────────────────────────────────
    import pickle
    model_path = output_path / 'rf_model_construction_start.pkl'
    with open(model_path, 'wb') as f:
        pickle.dump(clf, f)
    print(f'  Saved: {model_path.name}')

    return clf


In [ ]:
def main():
    print('=' * 70)
    print('NDVI TRAINING DATA ANALYSIS + THRESHOLD COMPUTATION (START + END)')
    print('=' * 70)

    output_path = Path(OUTPUT_DIR)
    output_path.mkdir(parents=True, exist_ok=True)

    # ── Load training labels ───────────────────────────────────────────────
    print('\nLoading training labels...')
    df_train = load_training_csv(TRAINING_CSV)

    print('\n  Category breakdown:')
    for cat, grp in df_train.groupby('category_start'):
        print(f'    start/{cat:<28} {len(grp)}')
    for cat, grp in df_train.groupby('category_end'):
        print(f'    end  /{cat:<28} {len(grp)}')

    # START map: labeled starts with a known year
    labeled_start = df_train[
        (df_train['category_start'] == 'labeled') & df_train['start_year'].notna()
    ]
    start_map = labeled_start.set_index('nda_id')['start_year'].astype(int).to_dict()

    uc_df = df_train[df_train['category_start'] == 'under_construction']

    # END map: confirmed ends only (censored excluded)
    labeled_end = df_train[
        (df_train['category_end'] == 'labeled') & df_train['end_year'].notna()
    ]
    end_map  = labeled_end.set_index('nda_id')['end_year'].astype(int).to_dict()
    null_ids = set(df_train[df_train['category_start'] == 'skip']['nda_id'])

    print(f'\n  {len(start_map)} NDAs with labeled start')
    print(f'  {len(end_map)}   NDAs with labeled end  (censored excluded)')
    print(f'  {len(uc_df)}     under_construction')
    print(f'  {len(null_ids)}  skip (already_built_up / hard_to_say)')

    # ── NDVI rasters ──────────────────────────────────────────────────────
    print('\nFinding NDVI rasters...')
    ndvi_tile_index = build_ndvi_tile_index(NDVI_DIR, NDVI_PATTERN)
    first_year      = min(ndvi_tile_index.keys())

    # Extend start_map with under_construction NDAs (use first available year)
    uc_map = (uc_df.dropna(subset=['start_year'])
                   .set_index('nda_id')['start_year'].astype(int).to_dict())
    for nda_id in set(uc_df['nda_id']) - set(uc_map.keys()):
        uc_map[nda_id] = first_year
    start_map_full = {**start_map, **uc_map}

    # ── Load geometries ────────────────────────────────────────────────────
    print('\nLoading NDA geometries...')
    gdf = gpd.read_file(INPUT_GPKG)
    if 'nhda_id' in gdf.columns and 'nda_id' not in gdf.columns:
        gdf = gdf.rename(columns={'nhda_id': 'nda_id'})

    all_ids  = set(start_map_full.keys()) | set(end_map.keys()) | null_ids
    gdf_sub  = gdf[gdf['nda_id'].isin(all_ids)].copy()
    print(f'  {len(gdf_sub)} training NDAs found in GPKG')

    # ── Extract NDVI time series ───────────────────────────────────────────
    print('\nExtracting NDVI time series...')
    ts_df = extract_all_time_series(gdf_sub, ndvi_tile_index)
    ts_df.to_csv(output_path / 'training_ndvi_timeseries.csv',
                 index=False, encoding='utf-8-sig')
    print(f'  {len(ts_df):,} records  →  training_ndvi_timeseries.csv')

    # Diagnostics
    print(f'  Rows with median=NaN: {ts_df["median"].isna().sum()}')
    print(f'  NDAs with >= 1 valid year: {ts_df.dropna(subset=["median"])["nda_id"].nunique()}')
    print('  Valid entries per year:')
    print(ts_df.groupby('year')['median'].apply(lambda x: x.notna().sum()).to_string())

    # ── Plots ──────────────────────────────────────────────────────────────
    print('\nGenerating heatmaps...')
    plot_heatmaps(ts_df, start_map_full, null_ids, output_path)

    np.random.seed(42)
    print('\nGenerating aligned plots — construction START...')
    plot_aligned_generic(
        ts_df, start_map, null_ids,
        WINDOW_BEFORE, WINDOW_AFTER,
        output_path,
        'aligned_ndvi_construction_start.png',
        'Construction Start'
    )

    print('\nGenerating aligned plots — construction END...')
    plot_aligned_generic(
        ts_df, end_map, null_ids,
        WINDOW_BEFORE, WINDOW_AFTER,
        output_path,
        'aligned_ndvi_construction_end.png',
        'Construction End'
    )

    # ── Thresholds ─────────────────────────────────────────────────────────
    print('\n' + '=' * 70)
    print('COMPUTING THRESHOLDS FROM RAW TIME SERIES')
    print('=' * 70)
    compute_thresholds(
        ts_df, start_map, end_map, null_ids, output_path,
        labeled_end=labeled_end
    )

    # ── Random Forest ──────────────────────────────────────────────────────
    print('\n' + '=' * 70)
    print('RANDOM FOREST — CONSTRUCTION START DETECTION')
    print('=' * 70)
    run_random_forest(ts_df, start_map_full, end_map, null_ids, output_path)

    print('\nDone!  All outputs saved to:')
    print(f'  {output_path}')


main()
